# PII Redaction with Spark NLP on SageMaker

This notebook demonstrates **PII (Personally Identifiable Information) redaction** using John Snow Labs' Spark NLP on Amazon SageMaker.

**Pipeline overview:**
1. Detect named entities (PERSON, DATE, LOCATION, ORG, etc.) using a pre-trained NER model
2. Mask/replace identified entities with their entity-type labels
3. Visualize redacted output

**SageMaker v2.x compatible** — uses `sagemaker>=2.0` SDK APIs.

## 1. Environment Setup

Spark NLP requires **Java 8 or 11**. The cell below installs Java and all required Python packages.

In [1]:
import subprocess, sys, os

# ── Install Java if not already present (required by PySpark / Spark NLP) ──
java_check = subprocess.run(["java", "-version"], capture_output=True)
if java_check.returncode != 0:
    print("Java not found — installing default-jdk ...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "default-jdk"], check=True)
    print("Java installed.")
else:
    print("Java already available:", java_check.stderr.decode().split('\n')[0])

# ── Set JAVA_HOME so PySpark can locate the JVM ──
java_home_candidates = [
    "/usr/lib/jvm/java-11-openjdk-amd64",
    "/usr/lib/jvm/java-11-amazon-corretto",
    "/usr/lib/jvm/java-8-openjdk-amd64",
    "/usr/lib/jvm/java-8-amazon-corretto",
]
for candidate in java_home_candidates:
    if os.path.isdir(candidate):
        os.environ["JAVA_HOME"] = candidate
        break
else:
    # Fall back to `java -XshowSettings:all` detection
    result = subprocess.run(
        ["java", "-XshowSettings:all", "-version"],
        capture_output=True, text=True
    )
    for line in result.stderr.splitlines():
        if "java.home" in line:
            java_home = line.split("=")[1].strip()
            # Strip trailing /jre if present
            if java_home.endswith("/jre"):
                java_home = java_home[:-4]
            os.environ["JAVA_HOME"] = java_home
            break

print("JAVA_HOME:", os.environ.get("JAVA_HOME", "NOT SET — set manually if Spark fails to start"))

Java already available: openjdk version "11.0.30" 2026-01-20
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


In [2]:
# Install Python packages
# Pin versions to known-good combinations for Spark NLP on SageMaker
!pip install -q \
    "sagemaker>=2.0,<3.0" \
    "pyspark>=3.3,<3.6" \
    "spark-nlp" \
    "spark-nlp-display" \
    "boto3" \
    "nltk"

print("All packages installed.")

All packages installed.


## 2. SageMaker v2.x Session Setup

In [3]:
import boto3
import sagemaker
from sagemaker import get_execution_role

# ── SageMaker v2.x session ──────────────────────────────────────────────────
# sagemaker.Session() automatically picks up the region from the instance
# metadata or from the boto3 session / environment variable AWS_DEFAULT_REGION.
boto_session = boto3.Session()
sm_session   = sagemaker.Session(boto_session=boto_session)

# Execution role — works both inside SageMaker and locally (with appropriate
# IAM permissions attached to your local credentials).
try:
    role = get_execution_role(sagemaker_session=sm_session)
except Exception:
    # Running locally without an instance profile — use an env var or hard-code
    role = os.environ.get("SAGEMAKER_ROLE", "<YOUR_SAGEMAKER_EXECUTION_ROLE_ARN>")
    print("WARNING: Could not auto-detect execution role. Set SAGEMAKER_ROLE env var.")

region     = boto_session.region_name or sm_session.boto_region_name
bucket     = sm_session.default_bucket()          # SageMaker default S3 bucket
prefix     = "spark-nlp/redaction"               # S3 key prefix for artefacts

print(f"SageMaker SDK version : {sagemaker.__version__}")
print(f"Region                : {region}")
print(f"Default S3 bucket     : {bucket}")
print(f"Execution role ARN    : {role}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/cjen/.config/sagemaker/config.yaml
SageMaker SDK version : 2.257.1
Region                : us-west-2
Default S3 bucket     : sagemaker-us-west-2-493644444178
Execution role ARN    : arn:aws:iam::493644444178:role/aws-reserved/sso.amazonaws.com/ap-southeast-1/AWSReservedSSO_AWSAdministratorAccess_e93147afb9349dfd


## 3. Import Spark NLP & Start Spark Session

In [4]:
import sys
print("Python executable:", sys.executable)

import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp.pretrained import PretrainedPipeline
from sparknlp_display import NerVisualizer

import pyspark.sql.functions as F
from pyspark.sql.types import StringType

import pandas as pd
import re
import warnings

warnings.filterwarnings("ignore")
print("Spark NLP version:", sparknlp.version())

Python executable: /home/cjen/mySageMaker/ML/spark-nlp/redaction/.venv/bin/python
Spark NLP version: 6.3.3


In [5]:
# ── Spark session parameters tuned for a SageMaker ml.m5.4xlarge (64 GB RAM) ─
# Adjust spark.driver.memory to ~75 % of available instance RAM.
params = {
    "spark.driver.memory"           : "16G",
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.driver.maxResultSize"    : "2000M",
    # Disable Spark UI to avoid port conflicts on SageMaker
    "spark.ui.enabled"              : "false",
}

spark = sparknlp.start(params=params)
spark

26/03/20 11:47:07 WARN Utils: Your hostname, MindyJen1008 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/20 11:47:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/cjen/mySageMaker/ML/spark-nlp/redaction/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/cjen/.ivy2/cache
The jars for the packages stored in: /home/cjen/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4bf512f5-b705-4050-8841-140734723732;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.3.3 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	f

## 4. Sample Data

We create a small DataFrame with clinical / HR text containing PII such as names, dates, and locations.

In [6]:
sample_texts = [
    "Patient John Smith, born on March 4 1975, was admitted to Montreal General Hospital on 2024-11-01 with chest pain.",
    "Dr. Emily Carter reviewed the case of Michael Johnson, SSN 123-45-6789, residing at 42 Maple Street, Toronto.",
    "The contract signed by Alice Wong and Bob Martin on January 15 2023 includes a clause for New York jurisdiction.",
    "Call center agent Sarah Lee (employee ID E-4892) handled complaint #CR-00312 on 09/22/2024 at 3:45 PM.",
    "Invoice #INV-2024-88 issued to Acme Corp, attention David Brown, 100 King Street West, Vancouver, BC V6B 1A1.",
]

pdf = pd.DataFrame(sample_texts, columns=["text"])
spark_df = spark.createDataFrame(pdf)
spark_df.show(truncate=False)

+------------------------------------------------------------------------------------------------------------------+
|text                                                                                                              |
+------------------------------------------------------------------------------------------------------------------+
|Patient John Smith, born on March 4 1975, was admitted to Montreal General Hospital on 2024-11-01 with chest pain.|
|Dr. Emily Carter reviewed the case of Michael Johnson, SSN 123-45-6789, residing at 42 Maple Street, Toronto.     |
|The contract signed by Alice Wong and Bob Martin on January 15 2023 includes a clause for New York jurisdiction.  |
|Call center agent Sarah Lee (employee ID E-4892) handled complaint #CR-00312 on 09/22/2024 at 3:45 PM.            |
|Invoice #INV-2024-88 issued to Acme Corp, attention David Brown, 100 King Street West, Vancouver, BC V6B 1A1.     |
+---------------------------------------------------------------

## 5. Redaction Pipeline

We use a pre-trained NER model (`onto_100`) that recognises:
`PERSON`, `DATE`, `TIME`, `CARDINAL`, `ORG`, `GPE`, `LOC`, `MONEY`, `PERCENT`, `PRODUCT`, `EVENT`, `NORP`, `FAC`, `QUANTITY`, `LAW`, `LANGUAGE`, `WORK_OF_ART`, `ORDINAL`

Each detected entity span is then replaced with `[ENTITY_TYPE]` to produce the redacted text.

In [7]:
# ── Stage 1: Raw text → Document annotation ──────────────────────────────────
document_assembler = (
    DocumentAssembler()
    .setInputCol("text")
    .setOutputCol("document")
)

# ── Stage 2: Sentence segmentation ──────────────────────────────────────────
sentence_detector = (
    SentenceDetector()
    .setInputCols(["document"])
    .setOutputCol("sentence")
)

# ── Stage 3: Tokenisation ────────────────────────────────────────────────────
tokenizer = (
    Tokenizer()
    .setInputCols(["sentence"])
    .setOutputCol("token")
)

# ── Stage 4: Word embeddings (GloVe 100d, ~130 MB) ───────────────────────────
# Downloaded automatically from the John Snow Labs public S3 bucket.
embeddings = (
    WordEmbeddingsModel.pretrained("glove_100d")
    .setInputCols(["sentence", "token"])
    .setOutputCol("embeddings")
)

# ── Stage 5: NER model trained on OntoNotes 5 (18-class NER) ─────────────────
ner = (
    NerDLModel.pretrained("ner_dl")
    .setInputCols(["sentence", "token", "embeddings"])
    .setOutputCol("ner")
)

# ── Stage 6: Convert BIO tags → entity chunks ────────────────────────────────
ner_converter = (
    NerConverter()
    .setInputCols(["sentence", "token", "ner"])
    .setOutputCol("ner_chunk")
)

# ── Assemble the pipeline ────────────────────────────────────────────────────
nlp_pipeline = Pipeline(
    stages=[
        document_assembler,
        sentence_detector,
        tokenizer,
        embeddings,
        ner,
        ner_converter,
    ]
)

print("Pipeline assembled — fitting (downloads models on first run) ...")

glove_100d download started this may take some time.
Approximate size to download 145.3 MB
[ | ]

26/03/20 11:47:48 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


glove_100d download started this may take some time.
Approximate size to download 145.3 MB
[ / ]

26/03/20 11:47:52 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


[ — ]Download done! Loading the resource.
[OK!]
ner_dl download started this may take some time.


...

Approximate size to download 13.6 MB
[ | ]

26/03/20 11:47:55 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/20 11:47:56 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


ner_dl download started this may take some time.
Approximate size to download 13.6 MB
Download done! Loading the resource.
[ / ]

2026-03-20 11:47:59.148880: I external/org_tensorflow/tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-20 11:47:59.246408: W external/org_tensorflow/tensorflow/core/common_runtime/colocation_graph.cc:1218] Failed to place the graph without changing the devices of some resources. Some of the operations (that had to be colocated with resource generating operations) are not supported on the resources' devices. Current candidate devices are [
  /job:localhost/replica:0/task:0/device:CPU:0].
See below for details of this colocation group:
Colocation Debug Info:
Colocation group had the following types and supported devices: 
Root Member(assigned_device_name_index_=-1 requested_device_name_='/device:GPU:0' assigned_device_nam

[OK!]
Pipeline assembled — fitting (downloads models on first run) ...


In [8]:
# Fit on an empty DataFrame (no training — just downloads/loads pretrained weights)
empty_df    = spark.createDataFrame([[""]], ["text"])
nlp_model   = nlp_pipeline.fit(empty_df)
print("Model ready.")

Model ready.


## 6. Run NER Inference & Inspect Raw Results

In [9]:
result_df = nlp_model.transform(spark_df)

# Flatten entity chunks for easy inspection
entities_df = result_df.select(
    F.col("text"),
    F.explode("ner_chunk").alias("chunk")
).select(
    "text",
    F.col("chunk.result").alias("entity_text"),
    F.col("chunk.metadata")["entity"].alias("entity_type"),
    F.col("chunk.begin").alias("begin"),
    F.col("chunk.end").alias("end"),
)

entities_df.show(30, truncate=80)

+--------------------------------------------------------------------------------+-------------------------+-----------+-----+---+
|                                                                            text|              entity_text|entity_type|begin|end|
+--------------------------------------------------------------------------------+-------------------------+-----------+-----+---+
|Patient John Smith, born on March 4 1975, was admitted to Montreal General Ho...|               John Smith|        PER|    8| 17|
|Patient John Smith, born on March 4 1975, was admitted to Montreal General Ho...|Montreal General Hospital|        LOC|   58| 82|
|Dr. Emily Carter reviewed the case of Michael Johnson, SSN 123-45-6789, resid...|             Emily Carter|        PER|    4| 15|
|Dr. Emily Carter reviewed the case of Michael Johnson, SSN 123-45-6789, resid...|          Michael Johnson|        PER|   38| 52|
|Dr. Emily Carter reviewed the case of Michael Johnson, SSN 123-45-6789, resid...| 

## 7. Apply Redaction

We replace each detected entity span with `[ENTITY_TYPE]` using a Python UDF.

In [11]:
from pyspark.sql.types import ArrayType, StructType, StructField, IntegerType

def redact_text(text, chunks):
    """Replace entity spans (sorted right-to-left) with [ENTITY_TYPE] tokens."""
    if not text or not chunks:
        return text
    # Sort descending by begin offset so replacements don't shift positions
    sorted_chunks = sorted(chunks, key=lambda c: c["begin"], reverse=True)
    text_list = list(text)
    for chunk in sorted_chunks:
        entity_type = chunk["metadata"]["entity"]
        replacement = f"[{entity_type}]"
        text_list[chunk["begin"]:chunk["end"] + 1] = list(replacement)
    return "".join(text_list)

redact_udf = F.udf(redact_text, StringType())

redacted_df = result_df.withColumn(
    "redacted_text", redact_udf(F.col("text"), F.col("ner_chunk"))
).select("text", "redacted_text")

redacted_df.show(truncate=False)

+------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|text                                                                                                              |redacted_text                                                                                      |
+------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|Patient John Smith, born on March 4 1975, was admitted to Montreal General Hospital on 2024-11-01 with chest pain.|Patient [PER], born on March 4 1975, was admitted to [LOC] on 2024-11-01 with chest pain.          |
|Dr. Emily Carter reviewed the case of Michael Johnson, SSN 123-45-6789, residing at 42 Maple Street, Toronto.     |Dr. [PER] review

## 8. HTML Visualization with NerVisualizer

`NerVisualizer` from `spark-nlp-display` renders entity spans inline with colour-coded labels.

In [12]:
from IPython.display import display, HTML

visualizer = NerVisualizer()

# Use LightPipeline for single-row annotation (faster for small inputs)
light_model = LightPipeline(nlp_model)

for text in sample_texts:
    annotations = light_model.fullAnnotate(text)[0]
    # NerVisualizer.display() returns an HTML string
    html = visualizer.display(
        annotations,
        label_col="ner_chunk",
        document_col="document",
        return_html=True,
    )
    display(HTML(html))
    print()

## 9. (Optional) Save Redacted Output to S3

Uses the **SageMaker v2.x** `sagemaker.s3.S3Uploader` API (replaces deprecated `sagemaker.Session.upload_data` in v1).

In [13]:
import os, tempfile
from sagemaker.s3 import S3Uploader          # SageMaker v2.x API

# Write redacted results as CSV locally, then upload to S3
local_output = os.path.join(tempfile.mkdtemp(), "redacted_output.csv")
redacted_df.toPandas().to_csv(local_output, index=False)

s3_uri = f"s3://{bucket}/{prefix}/redacted_output.csv"

# S3Uploader.upload is the SageMaker v2.x replacement for
# the v1 sagemaker.Session.upload_data() method.
S3Uploader.upload(
    local_path=local_output,
    desired_s3_uri=s3_uri,
    sagemaker_session=sm_session,
)
print(f"Redacted output uploaded to: {s3_uri}")

Redacted output uploaded to: s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/redacted_output.csv


## 10. Cleanup

In [14]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
